In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = []
# ///

In [ ]:
import importlib
import tqdm
importlib.reload(tqdm)

In [ ]:
import os
from glob import glob
from datetime import timedelta
from tqdm import tqdm

import pandas as pd
import numpy as np

from scipy.interpolate import interp1d

from functools import partial


# 本地文件数据分析 

In [ ]:
data_dir = 'data/atm'

var_name_list = ['wind', 'humidity', 'temperature', 'visibility']

for i, var in enumerate(var_name_list):

    print(i, var)
    file = f'{data_dir}/{var}.csv'

    df = pd.read_csv(file)
    drop_columns = set(['avg','40m', '80m']).intersection(set(df.columns))
    df.drop(drop_columns, axis=1, inplace=True)
    df['time'] = pd.to_datetime(df['time'])
    data = []

    for start_time in target_time_df.time.to_list():
        sub_df = df[(df['time'] > start_time) & (df['time'] < (start_time+timedelta(minutes=1)))]
        position = [int(index_str[:-1]) for index_str in sub_df.columns if index_str[:-1].isdigit()]
        sub_df = sub_df[[c for c in sub_df.columns if c[:-1].isdigit()]].mode().iloc[0]

        position = np.array(position)
        gap = np.min(position[1:]-position[:-1])
        equal_interval_values = interp1d(position, sub_df.values, kind='linear')(np.arange(np.min(position),np.max(position),gap))
        avg_value = np.mean(equal_interval_values)

        data.append((start_time, avg_value))

    res_df = pd.DataFrame(data, columns=['time', 'value'])
    res_df.to_csv(f'data/res/{var}.csv', index=False)

In [ ]:
r0_df = pd.read_csv(f'{data_dir}/r0_computed.csv')

In [ ]:
sub_df

In [ ]:
r0_df['time'] = pd.to_datetime(r0_df['time'])
data = []

for start_time in target_time_df.time.to_list():
    sub_df = r0_df[(r0_df['time'] > start_time) & (r0_df['time'] < (start_time+timedelta(minutes=1)))]
    sub_df = sub_df.mode().iloc[0].values[0]
    data.append((start_time, sub_df))

res_df = pd.DataFrame(data, columns=['time', 'value'])
res_df.to_csv('data/res/r0_computed.csv', index=False)

In [ ]:
position = np.array(position)
gap = np.min(position[1:]-position[:-1])
equal_interval_values = interp1d(position, sub_df.values, kind='linear')(np.arange(np.min(position),np.max(position),gap))
np.mean(equal_interval_values)

# 数据服务器上的数据分析

In [ ]:
variances_name = os.listdir('../data/20250620/24LiveStatisticData/atmosphere')
variances_name

In [ ]:
for i, var in enumerate(variances_name):
    print(f'{i+1}/{len(variances_name)}: {var}')
    
    raw_files = glob(f'/home/cdgds/workspace/data/*/24LiveStatisticData/atmosphere/{var}/{var}.xlsx')
    if raw_files:
        print(f'files counts: {len(raw_files)}')
        merged_df = pd.concat([pd.read_excel(f) for f in raw_files], axis=0)
        unit_col_name = merged_df.columns[0]
        
        merged_df['time'] = pd.to_datetime('2025-'+merged_df[unit_col_name])
        merged_df.drop(unit_col_name, axis=1, inplace=True)
        merged_df.to_csv(f'./data/{var.lower()}.csv', index=False)


# 平均风速 平均温度 平均湿度 平均能见度 计算r0

In [ ]:
target_time =['2025/6/6\t10:24', '2025/6/6\t10:36', '2025/6/6\t10:45', '2025/6/6\t10:55', '2025/6/6\t11:26', '2025/6/6\t11:37', '2025/6/6\t14:29', '2025/6/6\t14:49', '2025/6/9\t11:42', '2025/6/9\t11:47', '2025/6/9\t13:26', '2025/6/9\t13:36', '2025/6/9\t13:46', '2025/6/9\t15:09', '2025/6/9\t15:18', '2025/6/10\t13:45', '2025/6/10\t13:52', '2025/6/10\t14:57', '2025/6/10\t15:06', '2025/6/10\t15:56', '2025/6/10\t16:02', '2025/6/11\t16:35', '2025/6/11\t16:47', '2025/6/11\t16:58', '2025/6/12\t9:04', '2025/6/12\t9:13', '2025/6/12\t9:17', '2025/6/12\t9:39', '2025/6/12\t9:42', '2025/6/12\t11:29', '2025/6/12\t11:35']

target_time_df = pd.DataFrame(target_time, columns=['time'])
target_time_df['time'] = pd.to_datetime(target_time_df['time'], format="%Y/%m/%d\t%H:%M")

target_time_df

In [ ]:
def avg(position, value):
    position = np.array(position)
    gap = np.min(position[1:]-position[:-1])
    equal_interval_values = interp1d(position, value, kind='linear')(np.arange(np.min(position),np.max(position),gap))
    return np.mean(equal_interval_values)

In [ ]:
var_name_list = ['wind', 'humidity', 'temperature', 'visibility']
for name in var_name_list:
    file = f'data/{name}.csv'
    df = pd.read_csv(file)
    df['time'] = pd.to_datetime(df['time'])
    # 按分钟截断
    df['minute'] = df['time'].dt.floor('T')
    
    try:
        df = df.drop('40m', axis=1).drop('80m', axis=1)
    except:
        pass
        
    # 定义众数函数
    def mode_value(x):
        return x.mode().iloc[0] if not x.mode().empty else None
    
    # 聚合
    df = df.groupby('minute').agg(mode_value).reset_index()
    position = [int(p[:-1]) for p in df.columns if p[:-1].isdigit()]
    if len(position) > 1:
        df_avg_func = partial(avg, position)
        df['avg'] = df[[f'{p}m' for p in position]].apply(df_avg_func, axis=1)

        df.to_csv(f'data/{name}-minute.csv', index=False)